# 가중 교차엔트로피 U-Net — 클래스 불균형 분할 (Colab)

**연습 기법** (IOAI 2025 *at-home* **Radar**(개별) 와 동일): **다중클래스 U-Net** 을 **가중 교차엔트로피**
(weighted CrossEntropyLoss)로 학습해 **심한 클래스 불균형**을 이겨낸다. Radar 모범답안2 가 이 기법으로
0.7963 → **0.9834** 로 뛰었다(희소한 타깃 픽셀이 거대한 배경에 묻히는 문제 → 역빈도 클래스 가중으로 해결).

**대회**: [Data Science Bowl 2018](https://www.kaggle.com/c/data-science-bowl-2018) (핵 분할). 09번(이진 BCE+Dice)
과 달리, 마스크를 **3클래스**(0=배경 · 1=핵 코어 · 2=핵 경계)로 바꾼다. 각 핵을 **침식**해 남긴
**코어(class 1)가 전체의 ~3%** 로 *희소*하다 → 손실 가중이 의미 있는 상황.

**핵심 흐름 & 배울 점**:
1. 인스턴스 마스크 → 침식으로 **희소 코어 / 경계** 분리 → 3클래스 라벨맵.
2. **세 가지 손실 비교** — 무가중 vs **역빈도 가중(raw)** vs **역제곱근 가중(sqrt)**.
3. 교훈: 가중은 희소 클래스(코어) IoU 를 살리지만, **raw 역빈도는 너무 공격적**이라 배경을 망쳐 역효과 →
   **부드러운 sqrt 가중이 가장 안정적으로 mIoU 를 끌어올린다**(가중치 *스케일*이 핵심).
4. 살린 **코어를 마커**로 **watershed** → 맞닿은 핵을 분리한 인스턴스 RLE 제출(09 대비 개선).

> ⚙️ **GPU 권장**: 런타임 → 런타임 유형 변경 → GPU. (작은 U-Net 3회 학습이라 T4 에서 ~10분)
> ⚠️ 노트북에 개인 API 토큰이 평문으로 들어 있습니다. 외부에 공유/업로드하지 마세요.
> 💡 09번(이진 BCE+Dice)과 비교하면 *손실/불균형 처리*의 차이를 볼 수 있습니다.

## 0. 라이브러리 설치 & Kaggle 자격증명

In [ ]:
import sys, subprocess
for pkg in ["kaggle", "torch", "numpy", "scipy", "pillow", "scikit-image", "matplotlib"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)
print("라이브러리 준비 완료")

In [ ]:
import os
os.environ["KAGGLE_API_TOKEN"] = "KGAT_5dd261fded8e0d7eb2c29d8d65dfabea"
print("Kaggle 자격증명 설정 완료 (내장 토큰)")

## 1. Kaggle 에서 데이터 다운로드 (stage1_train / stage2_test_final)

In [ ]:
import os, zipfile
WORK_DIR = "/content" if os.path.isdir("/content") else os.getcwd()
from kaggle.api.kaggle_api_extended import KaggleApi
api = KaggleApi(); api.authenticate()
for fn in ["stage1_train.zip", "stage2_test_final.zip"]:
    api.competition_download_file("data-science-bowl-2018", fn, path=WORK_DIR, quiet=False)
for fn, out in [("stage1_train.zip", "stage1_train"), ("stage2_test_final.zip", "stage2_test_final")]:
    zp = os.path.join(WORK_DIR, fn)
    with zipfile.ZipFile(zp) as zf: zf.extractall(os.path.join(WORK_DIR, out))
    os.remove(zp)
TRAIN_DIR = os.path.join(WORK_DIR, "stage1_train")
TEST_DIR  = os.path.join(WORK_DIR, "stage2_test_final")   # 채점 기준 테스트셋
print("train samples:", len(os.listdir(TRAIN_DIR)), "| test samples:", len(os.listdir(TEST_DIR)))

## 2. 3클래스 라벨맵 만들기 — 배경 / 희소 코어 / 경계
각 핵 인스턴스 마스크를 **침식(2회)** 해 남긴 **코어(class 1)** 와, 원본−코어로 **경계(class 2)** 를 만든다.
코어는 전체의 약 **3%** 로 희소해 *클래스 불균형*을 만든다 → 가중 CE 가 필요한 상황을 재현.

In [ ]:
import numpy as np, glob
from PIL import Image
from scipy.ndimage import binary_erosion

SZ = 128
ERODE_ITERS = 2                         # 침식 → 코어를 희소하게(불균형 강화)
STRUCT = np.ones((3, 3), dtype=bool)

def load_sample(sample_dir):
    img_path = glob.glob(os.path.join(sample_dir, "images", "*.png"))[0]
    img = np.asarray(Image.open(img_path).convert("RGB").resize((SZ, SZ)), dtype="float32") / 255.0
    lab = np.zeros((SZ, SZ), dtype="int64")          # 0=배경
    for mp in glob.glob(os.path.join(sample_dir, "masks", "*.png")):
        m = np.asarray(Image.open(mp).convert("L").resize((SZ, SZ), Image.NEAREST)) > 0
        if m.sum() == 0:
            continue
        core = binary_erosion(m, STRUCT, iterations=ERODE_ITERS)
        lab[m] = 2                                    # 2=핵 경계
        lab[core] = 1                                 # 1=핵 코어 (희소, 코어가 경계를 덮어씀)
    return img.transpose(2, 0, 1), lab

ids = sorted(os.listdir(TRAIN_DIR))
X = np.stack([load_sample(os.path.join(TRAIN_DIR, i))[0] for i in ids])
Y = np.stack([load_sample(os.path.join(TRAIN_DIR, i))[1] for i in ids])
counts = np.bincount(Y.reshape(-1), minlength=3)
print("X:", X.shape, "| Y:", Y.shape)
print("픽셀 비율  배경:%.3f 코어:%.3f 경계:%.3f" % tuple(counts / counts.sum()))

## 3. 클래스 가중치 두 가지 & train/valid 분할
**raw 역빈도** `1/freq` 는 희소 코어를 크게 키우지만 배경을 0.09 배로 깎아 *과하게 공격적*이다.
**sqrt 역빈도** `1/√freq` 는 더 부드러워 배경을 덜 망친다 → 보통 더 안정적이다.

In [ ]:
import torch
freq = counts / counts.sum()
def norm(w): return torch.tensor((w / w.mean()).astype("float32"))
w_raw  = norm(1.0 / freq)            # 공격적
w_sqrt = norm(1.0 / np.sqrt(freq))   # 부드러움
print("raw  역빈도 가중 [배경,코어,경계]:", np.round(w_raw.numpy(), 2))
print("sqrt 역빈도 가중 [배경,코어,경계]:", np.round(w_sqrt.numpy(), 2))

rng = np.random.RandomState(42); idx = rng.permutation(len(X)); cut = int(len(X) * 0.9)
tr_i, va_i = idx[:cut], idx[cut:]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
Xt = torch.from_numpy(X); Yt = torch.from_numpy(Y)
print("device:", device, "| train:", len(tr_i), "valid:", len(va_i))

## 4. U-Net (다중클래스, n_classes=3) 정의

In [ ]:
import torch.nn as nn

class DoubleConv(nn.Module):
    def __init__(self, i, o):
        super().__init__()
        self.c = nn.Sequential(
            nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(inplace=True),
            nn.Conv2d(o, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.c(x)

class UNet(nn.Module):
    def __init__(self, ch=3, n_classes=3):
        super().__init__()
        self.d1 = DoubleConv(ch, 16); self.d2 = DoubleConv(16, 32); self.d3 = DoubleConv(32, 64)
        self.pool = nn.MaxPool2d(2); self.bottleneck = DoubleConv(64, 128)
        self.u3 = nn.ConvTranspose2d(128, 64, 2, 2); self.c3 = DoubleConv(128, 64)
        self.u2 = nn.ConvTranspose2d(64, 32, 2, 2); self.c2 = DoubleConv(64, 32)
        self.u1 = nn.ConvTranspose2d(32, 16, 2, 2); self.c1 = DoubleConv(32, 16)
        self.out = nn.Conv2d(16, n_classes, 1)
    def forward(self, x):
        c1 = self.d1(x); c2 = self.d2(self.pool(c1)); c3 = self.d3(self.pool(c2))
        b = self.bottleneck(self.pool(c3))
        x = self.c3(torch.cat([self.u3(b), c3], 1))
        x = self.c2(torch.cat([self.u2(x), c2], 1))
        x = self.c1(torch.cat([self.u1(x), c1], 1))
        return self.out(x)

print("U-Net 파라미터:", sum(p.numel() for p in UNet().parameters()))

## 5. 학습 — 무가중 vs raw 가중 vs sqrt 가중
동일한 U-Net·에폭·옵티마이저·seed 로, **CrossEntropyLoss 의 `weight` 인자만** 바꿔 효과를 분리한다.
평가지표는 Radar 처럼 **per-class IoU** 와 **mean IoU**.

In [ ]:
EPOCHS = 15; BS = 16

def iou_per_class(pred, gt, n=3):
    out = []
    for c in range(n):
        p, g = (pred == c), (gt == c)
        inter = (p & g).sum(); union = (p | g).sum()
        out.append(float(inter) / float(union) if union > 0 else float("nan"))
    return out

def train_unet(weight=None, seed=0):
    torch.manual_seed(seed)
    model = UNet().to(device)
    ce = nn.CrossEntropyLoss(weight=None if weight is None else weight.to(device))
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    for epoch in range(EPOCHS):
        model.train(); perm = np.random.RandomState(epoch).permutation(tr_i)
        for i in range(0, len(perm), BS):
            b = perm[i:i+BS]
            xb, yb = Xt[b].to(device), Yt[b].to(device)
            opt.zero_grad(); loss = ce(model(xb), yb); loss.backward(); opt.step()
    model.eval(); preds, gts = [], []
    with torch.no_grad():
        for j in range(0, len(va_i), BS):
            b = va_i[j:j+BS]
            preds.append(model(Xt[b].to(device)).argmax(1).cpu().numpy()); gts.append(Yt[b].numpy())
    return model, iou_per_class(np.concatenate(preds), np.concatenate(gts))

_,     iou_none = train_unet(weight=None)
_,     iou_raw  = train_unet(weight=w_raw)
model, iou_sqrt = train_unet(weight=w_sqrt)   # sqrt 모델을 최종 사용

def show(name, iou):
    print(f"{name:14s} | 배경 {iou[0]:.3f}  코어 {iou[1]:.3f}  경계 {iou[2]:.3f}  | mIoU {np.nanmean(iou):.3f}")
print("per-class IoU (배경/코어/경계):")
show("무가중 CE",        iou_none)
show("raw 역빈도 가중",  iou_raw)
show("sqrt 역빈도 가중", iou_sqrt)
print("\n→ raw 가중은 배경을 과도하게 깎아 mIoU 가 오히려 떨어질 수 있다(공격적).")
print("   sqrt 가중이 희소 '코어' IoU 를 안정적으로 올려 mIoU 개선 — Radar 의 weighted-CE 효과와 동형.")

## 6. 예측 시각화 (sqrt 가중 모델)

In [ ]:
import matplotlib.pyplot as plt
model.eval(); j = int(va_i[0])
with torch.no_grad():
    pr = model(Xt[j:j+1].to(device)).argmax(1)[0].cpu().numpy()
fig, ax = plt.subplots(1, 3, figsize=(12, 4))
ax[0].imshow(X[j].transpose(1, 2, 0)); ax[0].set_title("Image")
ax[1].imshow(Y[j], vmin=0, vmax=2); ax[1].set_title("GT (0=bg/1=core/2=edge)")
ax[2].imshow(pr, vmin=0, vmax=2); ax[2].set_title("Pred (0/1/2)")
for a in ax: a.axis("off")
plt.show()

## 7. 캐글 제출 — 코어를 마커로 watershed → 인스턴스 RLE
**코어(`pred==1`)** 의 연결요소를 **watershed 마커**로 쓰고, 핵 영역(`pred>=1`) 안에서 영역을 키워
맞닿은 핵을 **분리**한다(09 의 단순 라벨링 대비 개선). DSB2018 은 인스턴스별 RLE 행을 요구한다.

In [ ]:
import pandas as pd
from scipy import ndimage as ndi
from skimage.segmentation import watershed
from skimage.morphology import label

def rle_encode(mask):
    pixels = mask.flatten(order="F")            # DSB2018: 열 우선
    pixels = np.concatenate([[0], pixels, [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[::2]
    return " ".join(str(x) for x in runs)

def instances_from_pred(pr):
    full = pr >= 1                              # 코어+경계 = 핵 전체
    markers = label(pr == 1)                    # 희소 코어 = 마커(서로 떨어져 있어 핵을 가른다)
    if markers.max() == 0:
        return label(full)                      # 코어 없으면 통째로 라벨
    dist = ndi.distance_transform_edt(full)
    return watershed(-dist, markers, mask=full)

test_ids = sorted(os.listdir(TEST_DIR)); rows = []
model.eval()
for sid in test_ids:
    paths = glob.glob(os.path.join(TEST_DIR, sid, "images", "*.png"))
    found = False
    if paths:
        orig = Image.open(paths[0]).convert("RGB"); W, H = orig.size
        x = np.asarray(orig.resize((SZ, SZ)), dtype="float32").transpose(2, 0, 1) / 255.0
        with torch.no_grad():
            pr = model(torch.from_numpy(x[None]).to(device)).argmax(1)[0].cpu().numpy()
        inst = instances_from_pred(pr)
        inst = np.asarray(Image.fromarray(inst.astype("int32")).resize((W, H), Image.NEAREST))
        for k in range(1, int(inst.max()) + 1):
            rle = rle_encode(inst == k)
            if rle:
                rows.append({"ImageId": sid, "EncodedPixels": rle}); found = True
    if not found:
        rows.append({"ImageId": sid, "EncodedPixels": "1 1"})   # ID 누락 방지용 더미

submission_path = os.path.join(WORK_DIR, "submission.csv")
pd.DataFrame(rows, columns=["ImageId", "EncodedPixels"]).to_csv(submission_path, index=False)
print("Saved:", submission_path, "| rows:", len(rows),
      "| unique ids:", pd.DataFrame(rows)["ImageId"].nunique(), "/", len(test_ids))

In [ ]:
try:
    from google.colab import files
    files.download(submission_path)
except Exception as e:
    print("자동 다운로드 불가:", e)
    print("submission.csv 위치:", submission_path)

## 🏁 제출하기
`submission.csv` 를 [Data Science Bowl 2018](https://www.kaggle.com/c/data-science-bowl-2018/submit) 에 제출하세요.

**기법 요약** (IOAI 2025 Radar): 다중클래스 U-Net + **가중 교차엔트로피**로 *클래스 불균형*을 극복 — 희소
클래스(여기선 핵 코어)의 IoU 를 살려 mIoU 를 끌어올린다(Radar 모범답안2 의 0.7963→0.9834 와 동형). 단,
**가중치 스케일이 중요** — raw 역빈도는 너무 공격적이라 배경을 망쳐 역효과, **sqrt 역빈도가 안정적**이다.
더 끌어올리려면: median-frequency balancing, CE+Dice 혼합, 침식 두께(ERODE_ITERS)·해상도(SZ)↑, 증강(flip/rotate).